In [1]:
# Keep repository-relative paths valid from notebook subfolders.
from pathlib import Path
import os

os.chdir(next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "notebooks").is_dir() and (root / "requirements.txt").is_file()
))

# Resolve legacy IDX paths stored in existing CSVs without rewriting the data.
def _relocated_idx_path(value):
    text = str(value).replace("\\", "/")
    old_repo = "AI-Builders-Hackhaton-2026-Backend/"
    if old_repo in text:
        text = text.split(old_repo, 1)[1]
    old_raw = "data/idx_financial_statements/"
    if text.startswith(old_raw):
        text = "data/idx_financial/raw/" + text[len(old_raw):]
    return Path(text)

# CELL 1 - IMPORT AND LOAD FINAL NORMALIZED IDX DATASET

from pathlib import Path

import numpy as np
import pandas as pd


FINAL_NORMALIZED_FILE = Path(
    "data/idx_financial/normalized/idx_financial_metrics_normalized_final.csv"
)


idx_normalized_df = pd.read_csv(
    FINAL_NORMALIZED_FILE,
    low_memory=False
)


print(
    "Loaded:",
    FINAL_NORMALIZED_FILE
)

print(
    "Rows:",
    len(idx_normalized_df)
)

print(
    "Unique tickers:",
    idx_normalized_df[
        "ticker"
    ].nunique()
)

print(
    "Unique source files:",
    idx_normalized_df[
        "source_file"
    ].nunique()
)

Loaded: data\idx_financial_metrics_normalized_final.csv
Rows: 83442
Unique tickers: 924
Unique source files: 14755


In [2]:
# CELL 2 - VALIDATE FINAL NORMALIZED DATASET

required_columns = [
    "ticker",
    "year",
    "quarter",
    "metric",
    "normalized_value",
    "currency",
    "source_file"
]


missing_columns = [
    column
    for column in required_columns
    if column not in idx_normalized_df.columns
]


print(
    "Missing required columns:",
    missing_columns
)


print(
    "\nMissing normalized values:",
    idx_normalized_df[
        "normalized_value"
    ].isna().sum()
)


print(
    "\nDuplicate ticker/year/quarter/metric:",
    idx_normalized_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric"
        ]
    ].duplicated().sum()
)


print("\nMETRIC COUNTS")

print(
    idx_normalized_df[
        "metric"
    ].value_counts()
)


print("\nCURRENCY COUNTS")

print(
    idx_normalized_df[
        "currency"
    ].value_counts(
        dropna=False
    )
)

Missing required columns: []

Missing normalized values: 4

Duplicate ticker/year/quarter/metric: 0

METRIC COUNTS
metric
total_assets           14754
total_liabilities      14754
operating_cash_flow    14738
cash                   13779
revenue                13003
gross_profit           12414
Name: count, dtype: int64

CURRENCY COUNTS
currency
IDR    73034
USD    10404
NaN        4
Name: count, dtype: int64


In [3]:
# CELL 3 - PIVOT NORMALIZED METRICS TO TICKER-PERIOD LEVEL

metric_wide_df = (
    idx_normalized_df
    .pivot_table(
        index=[
            "ticker",
            "year",
            "quarter"
        ],
        columns="metric",
        values="normalized_value",
        aggfunc="first"
    )
    .reset_index()
)


metric_wide_df.columns.name = None


print(
    "Ticker-period rows:",
    len(metric_wide_df)
)

print(
    "Unique tickers:",
    metric_wide_df[
        "ticker"
    ].nunique()
)


print("\nCOLUMNS")

print(
    metric_wide_df.columns.tolist()
)


display(
    metric_wide_df.head(10)
)

Ticker-period rows: 14753
Unique tickers: 924

COLUMNS
['ticker', 'year', 'quarter', 'cash', 'gross_profit', 'operating_cash_flow', 'revenue', 'total_assets', 'total_liabilities']


,ticker,year,quarter,cash,gross_profit,operating_cash_flow,revenue,total_assets,total_liabilities
0,AADI,2024,Q4,1.518688e+09,1.465951e+09,1.198515e+09,5.319582e+09,5.992658e+09,2.629176e+09
1,AADI,2025,Q1,1.358333e+09,3.474080e+08,3.266550e+08,1.164437e+09,5.827769e+09,2.339594e+09
2,AALI,2020,Q1,1.470866e+12,9.267910e+11,6.606100e+11,4.796084e+12,2.921860e+13,9.854949e+12
3,AALI,2020,Q2,1.152117e+12,1.303517e+12,1.573827e+12,9.081017e+12,2.738127e+13,8.061912e+12
4,AALI,2020,Q4,9.788920e+11,2.962891e+12,2.322164e+12,1.880704e+13,2.778123e+13,8.533437e+12
5,AALI,2021,Q1,1.810842e+12,9.321850e+11,1.004935e+12,5.035167e+12,2.846902e+13,8.820539e+12
6,AALI,2021,Q2,2.532318e+12,2.214032e+12,2.399195e+12,1.083208e+13,2.868902e+13,8.741617e+12
7,AALI,2021,Q3,3.874116e+12,3.611195e+12,4.138836e+12,1.801447e+13,2.969401e+13,9.114312e+12
8,AALI,2021,Q4,3.896022e+12,4.830014e+12,4.895119e+12,2.432205e+13,3.039991e+13,9.228733e+12
9,AALI,2022,Q1,3.993025e+12,9.943630e+11,3.189140e+11,6.581321e+12,3.123278e+13,9.525261e+12


In [4]:
# CELL 4 - CALCULATE SAFE FINANCIAL RATIOS

benchmark_ratio_df = (
    metric_wide_df
    .copy()
)


def safe_divide(
    numerator,
    denominator
):
    numerator = pd.to_numeric(
        numerator,
        errors="coerce"
    )

    denominator = pd.to_numeric(
        denominator,
        errors="coerce"
    )

    result = np.where(
        (
            numerator.notna()
        )
        &
        (
            denominator.notna()
        )
        &
        (
            denominator != 0
        ),
        numerator / denominator,
        np.nan
    )

    return result


# Gross Margin
benchmark_ratio_df[
    "gross_margin"
] = safe_divide(
    benchmark_ratio_df.get(
        "gross_profit"
    ),
    benchmark_ratio_df.get(
        "revenue"
    )
)


# Debt to Assets
benchmark_ratio_df[
    "debt_to_assets"
] = safe_divide(
    benchmark_ratio_df.get(
        "total_liabilities"
    ),
    benchmark_ratio_df.get(
        "total_assets"
    )
)


# Cash to Assets
benchmark_ratio_df[
    "cash_to_assets"
] = safe_divide(
    benchmark_ratio_df.get(
        "cash"
    ),
    benchmark_ratio_df.get(
        "total_assets"
    )
)


# Operating Cash Flow to Revenue
benchmark_ratio_df[
    "ocf_to_revenue"
] = safe_divide(
    benchmark_ratio_df.get(
        "operating_cash_flow"
    ),
    benchmark_ratio_df.get(
        "revenue"
    )
)


ratio_columns = [
    "gross_margin",
    "debt_to_assets",
    "cash_to_assets",
    "ocf_to_revenue"
]


print("RATIO AVAILABILITY")

for ratio in ratio_columns:

    print(
        ratio,
        ":",
        benchmark_ratio_df[
            ratio
        ].notna().sum()
    )


display(
    benchmark_ratio_df[
        [
            "ticker",
            "year",
            "quarter",
            *ratio_columns
        ]
    ].head(20)
)

RATIO AVAILABILITY
gross_margin : 12349
debt_to_assets : 14753
cash_to_assets : 13778
ocf_to_revenue : 12963


,ticker,year,quarter,gross_margin,debt_to_assets,cash_to_assets,ocf_to_revenue
0,AADI,2024,Q4,0.275576,0.438733,0.253425,0.225302
1,AADI,2025,Q1,0.298348,0.401456,0.233079,0.280526
2,AALI,2020,Q1,0.193239,0.337283,0.050340,0.137739
3,AALI,2020,Q2,0.143543,0.294432,0.042077,0.173310
4,AALI,2020,Q4,0.157542,0.307166,0.035236,0.123473
5,AALI,2021,Q1,0.185135,0.309829,0.063607,0.199583
6,AALI,2021,Q2,0.204396,0.304703,0.088268,0.221490
7,AALI,2021,Q3,0.200461,0.306941,0.130468,0.229751
8,AALI,2021,Q4,0.198586,0.303578,0.128159,0.201263
9,AALI,2022,Q1,0.151089,0.304976,0.127847,0.048457


In [5]:
# CELL 5 - AUDIT CALCULATED RATIOS

print("RATIO SUMMARY")
print("=" * 80)


for ratio in ratio_columns:

    print(
        f"\n{ratio}"
    )

    print(
        benchmark_ratio_df[
            ratio
        ].describe(
            percentiles=[
                0.01,
                0.05,
                0.25,
                0.50,
                0.75,
                0.95,
                0.99
            ]
        )
    )


print("\nEXTREME RATIO COUNTS")


ratio_extreme_summary = []


for ratio in ratio_columns:

    series = (
        benchmark_ratio_df[
            ratio
        ]
        .dropna()
    )

    ratio_extreme_summary.append(
        {
            "ratio": ratio,

            "count":
                len(series),

            "below_-5":
                (series < -5).sum(),

            "below_-1":
                (series < -1).sum(),

            "above_1":
                (series > 1).sum(),

            "above_5":
                (series > 5).sum(),
        }
    )


ratio_extreme_summary_df = (
    pd.DataFrame(
        ratio_extreme_summary
    )
)


display(
    ratio_extreme_summary_df
)

RATIO SUMMARY

gross_margin
count    12349.000000
mean        -8.930782
std        607.639880
min     -57082.968401
1%          -1.547849
5%          -0.008469
25%          0.125477
50%          0.250933
75%          0.436881
95%          0.701540
99%          1.000000
max          1.142575
Name: gross_margin, dtype: float64

debt_to_assets
count     14753.000000
mean         26.345292
std        2874.198656
min           0.000008
1%            0.009191
5%            0.069758
25%           0.255320
50%           0.450129
75%           0.660714
95%           1.021891
99%           3.952491
max      348997.678386
Name: debt_to_assets, dtype: float64

cash_to_assets
count    1.377800e+04
mean     1.323173e+01
std      1.541740e+03
min      1.151412e-07
1%       2.231454e-04
5%       1.764924e-03
25%      1.506969e-02
50%      5.232138e-02
75%      1.336428e-01
95%      3.274664e-01
99%      6.099653e-01
max      1.809691e+05
Name: cash_to_assets, dtype: float64

ocf_to_revenue
count    12

,ratio,count,below_-5,below_-1,above_1,above_5
0,gross_margin,12349,49,146,3,0
1,debt_to_assets,14753,0,0,764,123
2,cash_to_assets,13778,0,0,1,1
3,ocf_to_revenue,12963,148,607,176,38


In [6]:
# CELL 6 - BUILD METRIC-LEVEL PROVENANCE FOR RATIO AUDIT

value_pivot_df = (
    idx_normalized_df
    .pivot_table(
        index=[
            "ticker",
            "year",
            "quarter"
        ],
        columns="metric",
        values="normalized_value",
        aggfunc="first"
    )
    .reset_index()
)

value_pivot_df.columns.name = None


source_pivot_df = (
    idx_normalized_df
    .pivot_table(
        index=[
            "ticker",
            "year",
            "quarter"
        ],
        columns="metric",
        values="source_file",
        aggfunc="first"
    )
    .reset_index()
)

source_pivot_df.columns.name = None


# Rename source columns so they do not collide
source_metric_columns = [
    column
    for column in source_pivot_df.columns
    if column not in [
        "ticker",
        "year",
        "quarter"
    ]
]

source_pivot_df = source_pivot_df.rename(
    columns={
        column: f"{column}_source_file"
        for column in source_metric_columns
    }
)


ratio_audit_df = (
    value_pivot_df
    .merge(
        source_pivot_df,
        on=[
            "ticker",
            "year",
            "quarter"
        ],
        how="left",
        validate="one_to_one"
    )
)


print(
    "Ratio audit ticker-period rows:",
    len(ratio_audit_df)
)

print(
    "Unique tickers:",
    ratio_audit_df[
        "ticker"
    ].nunique()
)


display(
    ratio_audit_df.head()
)

Ratio audit ticker-period rows: 14753
Unique tickers: 924


,ticker,year,quarter,cash,gross_profit,operating_cash_flow,revenue,total_assets,total_liabilities,cash_source_file,gross_profit_source_file,operating_cash_flow_source_file,revenue_source_file,total_assets_source_file,total_liabilities_source_file
0,AADI,2024,Q4,1.518688e+09,1.465951e+09,1.198515e+09,5.319582e+09,5.992658e+09,2.629176e+09,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...
1,AADI,2025,Q1,1.358333e+09,3.474080e+08,3.266550e+08,1.164437e+09,5.827769e+09,2.339594e+09,AADI_2025_Q1_FS.xlsx,AADI_2025_Q1_FS.xlsx,AADI_2025_Q1_FS.xlsx,AADI_2025_Q1_FS.xlsx,AADI_2025_Q1_FS.xlsx,AADI_2025_Q1_FS.xlsx
2,AALI,2020,Q1,1.470866e+12,9.267910e+11,6.606100e+11,4.796084e+12,2.921860e+13,9.854949e+12,AALI_2020_Q1_FS.xlsx,AALI_2020_Q1_FS.xlsx,AALI_2020_Q1_FS.xlsx,AALI_2020_Q1_FS.xlsx,AALI_2020_Q1_FS.xlsx,AALI_2020_Q1_FS.xlsx
3,AALI,2020,Q2,1.152117e+12,1.303517e+12,1.573827e+12,9.081017e+12,2.738127e+13,8.061912e+12,AALI_2020_Q2_FS.xlsx,AALI_2020_Q2_FS.xlsx,AALI_2020_Q2_FS.xlsx,AALI_2020_Q2_FS.xlsx,AALI_2020_Q2_FS.xlsx,AALI_2020_Q2_FS.xlsx
4,AALI,2020,Q4,9.788920e+11,2.962891e+12,2.322164e+12,1.880704e+13,2.778123e+13,8.533437e+12,AALI_2020_Q4_FS.xlsx,AALI_2020_Q4_FS.xlsx,AALI_2020_Q4_FS.xlsx,AALI_2020_Q4_FS.xlsx,AALI_2020_Q4_FS.xlsx,AALI_2020_Q4_FS.xlsx


In [7]:
# CELL 7 - CHECK SOURCE CONSISTENCY FOR EACH RATIO

ratio_audit_df[
    "gross_margin_same_source"
] = (
    ratio_audit_df[
        "gross_profit_source_file"
    ]
    ==
    ratio_audit_df[
        "revenue_source_file"
    ]
)


ratio_audit_df[
    "debt_to_assets_same_source"
] = (
    ratio_audit_df[
        "total_liabilities_source_file"
    ]
    ==
    ratio_audit_df[
        "total_assets_source_file"
    ]
)


ratio_audit_df[
    "cash_to_assets_same_source"
] = (
    ratio_audit_df[
        "cash_source_file"
    ]
    ==
    ratio_audit_df[
        "total_assets_source_file"
    ]
)


ratio_audit_df[
    "ocf_to_revenue_same_source"
] = (
    ratio_audit_df[
        "operating_cash_flow_source_file"
    ]
    ==
    ratio_audit_df[
        "revenue_source_file"
    ]
)


same_source_columns = [
    "gross_margin_same_source",
    "debt_to_assets_same_source",
    "cash_to_assets_same_source",
    "ocf_to_revenue_same_source"
]


print("SOURCE CONSISTENCY")

for column in same_source_columns:

    print(
        f"\n{column}"
    )

    print(
        ratio_audit_df[
            column
        ].value_counts(
            dropna=False
        )
    )

SOURCE CONSISTENCY

gross_margin_same_source
gross_margin_same_source
True     12374
False     2379
Name: count, dtype: int64

debt_to_assets_same_source
debt_to_assets_same_source
True    14753
Name: count, dtype: int64

cash_to_assets_same_source
cash_to_assets_same_source
True     13778
False      975
Name: count, dtype: int64

ocf_to_revenue_same_source
ocf_to_revenue_same_source
True     12990
False     1763
Name: count, dtype: int64


In [8]:
# CELL 8 - INSPECT EXTREME RATIO ROWS WITH PROVENANCE

ratio_audit_df[
    "gross_margin"
] = safe_divide(
    ratio_audit_df[
        "gross_profit"
    ],
    ratio_audit_df[
        "revenue"
    ]
)

ratio_audit_df[
    "debt_to_assets"
] = safe_divide(
    ratio_audit_df[
        "total_liabilities"
    ],
    ratio_audit_df[
        "total_assets"
    ]
)

ratio_audit_df[
    "cash_to_assets"
] = safe_divide(
    ratio_audit_df[
        "cash"
    ],
    ratio_audit_df[
        "total_assets"
    ]
)

ratio_audit_df[
    "ocf_to_revenue"
] = safe_divide(
    ratio_audit_df[
        "operating_cash_flow"
    ],
    ratio_audit_df[
        "revenue"
    ]
)


print("MOST EXTREME GROSS MARGIN")

display(
    ratio_audit_df[
        [
            "ticker",
            "year",
            "quarter",
            "gross_profit",
            "revenue",
            "gross_margin",
            "gross_profit_source_file",
            "revenue_source_file",
            "gross_margin_same_source"
        ]
    ]
    .sort_values(
        "gross_margin"
    )
    .head(20)
)


print("\nMOST EXTREME DEBT TO ASSETS")

display(
    ratio_audit_df[
        [
            "ticker",
            "year",
            "quarter",
            "total_liabilities",
            "total_assets",
            "debt_to_assets",
            "total_liabilities_source_file",
            "total_assets_source_file",
            "debt_to_assets_same_source"
        ]
    ]
    .sort_values(
        "debt_to_assets",
        ascending=False
    )
    .head(20)
)


print("\nMOST EXTREME CASH TO ASSETS")

display(
    ratio_audit_df[
        [
            "ticker",
            "year",
            "quarter",
            "cash",
            "total_assets",
            "cash_to_assets",
            "cash_source_file",
            "total_assets_source_file",
            "cash_to_assets_same_source"
        ]
    ]
    .sort_values(
        "cash_to_assets",
        ascending=False
    )
    .head(20)
)


print("\nMOST EXTREME OCF TO REVENUE")

ocf_extreme_df = (
    ratio_audit_df[
        [
            "ticker",
            "year",
            "quarter",
            "operating_cash_flow",
            "revenue",
            "ocf_to_revenue",
            "operating_cash_flow_source_file",
            "revenue_source_file",
            "ocf_to_revenue_same_source"
        ]
    ]
    .copy()
)

ocf_extreme_df[
    "abs_ocf_to_revenue"
] = (
    ocf_extreme_df[
        "ocf_to_revenue"
    ].abs()
)

display(
    ocf_extreme_df
    .sort_values(
        "abs_ocf_to_revenue",
        ascending=False
    )
    .head(20)
)

MOST EXTREME GROSS MARGIN


,ticker,year,quarter,gross_profit,revenue,gross_margin,gross_profit_source_file,revenue_source_file,gross_margin_same_source
5789,HDTX,2024,Q4,-3.071064e+10,538000.0,-57082.968401,HDTX_2024_Q4_FS.xlsx,HDTX_2024_Q4_FS.xlsx,True
5788,HDTX,2024,Q3,-1.834457e+10,538000.0,-34097.706320,HDTX_2024_Q3_FS.xlsx,HDTX_2024_Q3_FS.xlsx,True
5787,HDTX,2024,Q1,-6.065786e+09,538000.0,-11274.695167,HDTX_2024_Q1_FS.xlsx,HDTX_2024_Q1_FS.xlsx,True
7934,LAPD,2021,Q2,-1.591265e+10,8858115.0,-1796.392377,LAPD_2021_Q2_FS.xlsx,LAPD_2021_Q2_FS.xlsx,True
13411,TIRT,2023,Q4,-2.919396e+10,22360023.0,-1305.631795,TIRT_2023_Q4_FS.xlsx,TIRT_2023_Q4_FS.xlsx,True
5783,HDTX,2023,Q1,-6.848625e+09,5402000.0,-1267.794335,HDTX_2023_Q1_FS.xlsx,HDTX_2023_Q1_FS.xlsx,True
5784,HDTX,2023,Q2,-1.403996e+10,13241000.0,-1060.340080,HDTX_2023_Q2_FS.xlsx,HDTX_2023_Q2_FS.xlsx,True
5786,HDTX,2023,Q4,-2.948068e+10,27908000.0,-1056.352157,HDTX_2023_Q4_FS.xlsx,HDTX_2023_Q4_FS.xlsx,True
13410,TIRT,2023,Q3,-1.969968e+10,19272455.0,-1022.167599,TIRT_2023_Q3_FS.xlsx,TIRT_2023_Q3_FS.xlsx,True
7933,LAPD,2021,Q1,-7.992608e+09,8858115.0,-902.292168,LAPD_2021_Q1_FS.xlsx,LAPD_2021_Q1_FS.xlsx,True



MOST EXTREME DEBT TO ASSETS


,ticker,year,quarter,total_liabilities,total_assets,debt_to_assets,total_liabilities_source_file,total_assets_source_file,debt_to_assets_same_source
760,ANTM,2023,Q2,1.269258e+13,3.636867e+07,348997.678386,ANTM_2023_Q2_FS.xlsx,ANTM_2023_Q2_FS.xlsx,True
7939,LAPD,2022,Q3,2.623420e+11,6.916923e+07,3792.755752,LAPD_2022_Q3_FS.xlsx,LAPD_2022_Q3_FS.xlsx,True
7938,LAPD,2022,Q2,2.572563e+11,6.999136e+07,3675.544300,LAPD_2022_Q2_FS.xlsx,LAPD_2022_Q2_FS.xlsx,True
2877,BTEL,2020,Q4,1.130682e+13,3.266000e+09,3461.977648,BTEL_2020_Q4_FS.xlsx,BTEL_2020_Q4_FS.xlsx,True
7936,LAPD,2021,Q4,2.488226e+11,7.793900e+07,3192.529504,LAPD_2021_Q4_FS.xlsx,LAPD_2021_Q4_FS.xlsx,True
2878,BTEL,2021,Q1,1.133469e+13,3.817000e+09,2969.528163,BTEL_2021_Q1_FS.xlsx,BTEL_2021_Q1_FS.xlsx,True
7937,LAPD,2022,Q1,2.500337e+11,8.643155e+07,2892.852411,LAPD_2022_Q1_FS.xlsx,LAPD_2022_Q1_FS.xlsx,True
2876,BTEL,2020,Q3,9.672782e+12,4.544000e+09,2128.693222,BTEL_2020_Q3_FS.xlsx,BTEL_2020_Q3_FS.xlsx,True
2879,BTEL,2021,Q2,1.138309e+13,6.811000e+09,1671.279841,BTEL_2021_Q2_FS.xlsx,BTEL_2021_Q2_FS.xlsx,True
2874,BTEL,2020,Q1,1.550545e+13,1.605700e+10,965.650744,BTEL_2020_Q1_FS.xlsx,BTEL_2020_Q1_FS.xlsx,True



MOST EXTREME CASH TO ASSETS


,ticker,year,quarter,cash,total_assets,cash_to_assets,cash_source_file,total_assets_source_file,cash_to_assets_same_source
760,ANTM,2023,Q2,6.581605e+12,3.636867e+07,180969.106758,ANTM_2023_Q2_FS.xlsx,ANTM_2023_Q2_FS.xlsx,True
9976,OASA,2021,Q3,4.714548e+10,4.869313e+10,0.968216,OASA_2021_Q3_FS.xlsx,OASA_2021_Q3_FS.xlsx,True
9970,OASA,2020,Q1,4.790009e+10,4.960400e+10,0.965650,OASA_2020_Q1_FS.xlsx,OASA_2020_Q1_FS.xlsx,True
9973,OASA,2020,Q4,4.511678e+10,4.684005e+10,0.963209,OASA_2020_Q4_FS.xlsx,OASA_2020_Q4_FS.xlsx,True
9971,OASA,2020,Q2,4.568849e+10,4.787057e+10,0.954417,OASA_2020_Q2_FS.xlsx,OASA_2020_Q2_FS.xlsx,True
9972,OASA,2020,Q3,4.557329e+10,4.807902e+10,0.947883,OASA_2020_Q3_FS.xlsx,OASA_2020_Q3_FS.xlsx,True
2985,BUKA,2021,Q3,2.363788e+13,2.501976e+13,0.944768,BUKA_2021_Q3_FS.xlsx,BUKA_2021_Q3_FS.xlsx,True
517,ALKA,2023,Q4,3.161084e+11,3.397438e+11,0.930432,ALKA_2023_Q4_FS.xlsx,ALKA_2023_Q4_FS.xlsx,True
2986,BUKA,2021,Q4,2.470039e+13,2.661555e+13,0.928043,BUKA_2021_Q4_FS.xlsx,BUKA_2021_Q4_FS.xlsx,True
9975,OASA,2021,Q2,4.630111e+10,5.017127e+10,0.922861,OASA_2021_Q2_FS.xlsx,OASA_2021_Q2_FS.xlsx,True



MOST EXTREME OCF TO REVENUE


,ticker,year,quarter,operating_cash_flow,revenue,ocf_to_revenue,operating_cash_flow_source_file,revenue_source_file,ocf_to_revenue_same_source,abs_ocf_to_revenue
5788,HDTX,2024,Q3,-4.193269e+09,538000.0,-7794.180297,HDTX_2024_Q3_FS.xlsx,HDTX_2024_Q3_FS.xlsx,True,7794.180297
5789,HDTX,2024,Q4,-3.566482e+09,538000.0,-6629.148699,HDTX_2024_Q4_FS.xlsx,HDTX_2024_Q4_FS.xlsx,True,6629.148699
5787,HDTX,2024,Q1,3.790790e+08,538000.0,704.607807,HDTX_2024_Q1_FS.xlsx,HDTX_2024_Q1_FS.xlsx,True,704.607807
7934,LAPD,2021,Q2,-5.565587e+09,8858115.0,-628.303778,LAPD_2021_Q2_FS.xlsx,LAPD_2021_Q2_FS.xlsx,True,628.303778
13411,TIRT,2023,Q4,-1.216603e+10,22360023.0,-544.097520,TIRT_2023_Q4_FS.xlsx,TIRT_2023_Q4_FS.xlsx,True,544.097520
13410,TIRT,2023,Q3,-9.819694e+09,19272455.0,-509.519622,TIRT_2023_Q3_FS.xlsx,TIRT_2023_Q3_FS.xlsx,True,509.519622
10419,PGJO,2021,Q2,-3.600524e+09,7104300.0,-506.809064,PGJO_2021_Q2_FS.xlsx,PGJO_2021_Q2_FS.xlsx,True,506.809064
7933,LAPD,2021,Q1,-3.705881e+09,8858115.0,-418.359958,LAPD_2021_Q1_FS.xlsx,LAPD_2021_Q1_FS.xlsx,True,418.359958
13409,TIRT,2023,Q2,-6.889642e+09,19272455.0,-357.486477,TIRT_2023_Q2_FS.xlsx,TIRT_2023_Q2_FS.xlsx,True,357.486477
10418,PGJO,2021,Q1,-1.432002e+09,4768500.0,-300.304464,PGJO_2021_Q1_FS.xlsx,PGJO_2021_Q1_FS.xlsx,True,300.304464


In [9]:
# CELL 9 - AUDIT EXTREME RATIO CASES WITH NORMALIZATION DETAILS

extreme_files = [
    "ANTM_2023_Q2_FS.xlsx",
    "HDTX_2024_Q4_FS.xlsx",
    "LAPD_2022_Q3_FS.xlsx",
]

extreme_detail_df = (
    idx_normalized_df[
        idx_normalized_df[
            "source_file"
        ].isin(extreme_files)
    ]
    [
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "normalized_value",
            "currency",
            "unit_scale",
            "multiplier",
            "effective_multiplier",
            "final_scale_mode",
            "source_file",
            "source_sheet",
            "source_label",
            "currency_scale_status"
        ]
    ]
    .copy()
)


print("EXTREME FILE DETAIL")

display(
    extreme_detail_df
    .sort_values(
        [
            "ticker",
            "year",
            "quarter",
            "metric"
        ]
    )
)

EXTREME FILE DETAIL


,ticker,year,quarter,metric,selected_value,normalized_value,currency,unit_scale,multiplier,effective_multiplier,final_scale_mode,source_file,source_sheet,source_label,currency_scale_status
4175,ANTM,2023,Q2,cash,6.581605e+06,6.581605e+12,IDR,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,ANTM_2023_Q2_FS.xlsx,1210000,Kas dan setara kas,FOUND
4176,ANTM,2023,Q2,gross_profit,4.240811e+06,4.240811e+12,IDR,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,ANTM_2023_Q2_FS.xlsx,1311000,Jumlah laba bruto,FOUND
4177,ANTM,2023,Q2,operating_cash_flow,1.691955e+06,1.691955e+12,IDR,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,ANTM_2023_Q2_FS.xlsx,1510000,Total net cash flows received from (used in) o...,FOUND
4178,ANTM,2023,Q2,revenue,2.166111e+07,2.166111e+13,IDR,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,ANTM_2023_Q2_FS.xlsx,1311000,Sales and revenue,FOUND
4179,ANTM,2023,Q2,total_assets,3.636867e+01,3.636867e+07,IDR,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,ANTM_2023_Q2_FS.xlsx,1210000,Jumlah aset,FOUND
4180,ANTM,2023,Q2,total_liabilities,1.269258e+07,1.269258e+13,IDR,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,ANTM_2023_Q2_FS.xlsx,1210000,Jumlah liabilitas,FOUND
31683,HDTX,2024,Q4,cash,2.448360e+05,2.448360e+08,IDR,THOUSAND,1000.0,1000.0,PRESENTATION_SCALED,HDTX_2024_Q4_FS.xlsx,1210000,Kas dan setara kas,FOUND
31684,HDTX,2024,Q4,gross_profit,-3.071064e+07,-3.071064e+10,IDR,THOUSAND,1000.0,1000.0,PRESENTATION_SCALED,HDTX_2024_Q4_FS.xlsx,1311000,Jumlah laba bruto,FOUND
31685,HDTX,2024,Q4,operating_cash_flow,-3.566482e+06,-3.566482e+09,IDR,THOUSAND,1000.0,1000.0,PRESENTATION_SCALED,HDTX_2024_Q4_FS.xlsx,1510000,Total net cash flows received from (used in) o...,FOUND
31686,HDTX,2024,Q4,revenue,5.380000e+02,5.380000e+05,IDR,THOUSAND,1000.0,1000.0,PRESENTATION_SCALED,HDTX_2024_Q4_FS.xlsx,1311000,Sales and revenue,FOUND


In [11]:
# CELL 10 - COMPARE EXTREME FILES WITH NEIGHBOR PERIODS

extreme_tickers = (
    extreme_detail_df[
        "ticker"
    ]
    .dropna()
    .unique()
)


neighbor_audit_df = (
    idx_normalized_df[
        idx_normalized_df[
            "ticker"
        ].isin(
            extreme_tickers
        )
    ]
    [
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "normalized_value",
            "currency",
            "unit_scale",
            "multiplier",
            "effective_multiplier",
            "final_scale_mode",
            "source_file"
        ]
    ]
    .copy()
)


quarter_order = {
    "Q1": 1,
    "Q2": 2,
    "Q3": 3,
    "Q4": 4
}


neighbor_audit_df[
    "quarter_order"
] = (
    neighbor_audit_df[
        "quarter"
    ].map(
        quarter_order
    )
)


neighbor_audit_df = (
    neighbor_audit_df
    .sort_values(
        [
            "ticker",
            "metric",
            "year",
            "quarter_order"
        ]
    )
)


display(
    neighbor_audit_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "normalized_value",
            "unit_scale",
            "multiplier",
            "effective_multiplier",
            "final_scale_mode",
            "source_file"
        ]
    ]
)

,ticker,year,quarter,metric,selected_value,normalized_value,unit_scale,multiplier,effective_multiplier,final_scale_mode,source_file
4097,ANTM,2020,Q1,cash,3.160647e+09,3.160647e+12,THOUSAND,1000.0,1000.0,PRESENTATION_SCALED,ANTM_2020_Q1_FS.xlsx
4103,ANTM,2020,Q2,cash,3.006270e+09,3.006270e+12,THOUSAND,1000.0,1000.0,PRESENTATION_SCALED,ANTM_2020_Q2_FS.xlsx
4109,ANTM,2020,Q3,cash,3.669383e+09,3.669383e+12,THOUSAND,1000.0,1000.0,PRESENTATION_SCALED,ANTM_2020_Q3_FS.xlsx
4115,ANTM,2020,Q4,cash,3.984388e+09,3.984388e+12,THOUSAND,1000.0,1000.0,PRESENTATION_SCALED,ANTM_2020_Q4_FS.xlsx
4121,ANTM,2021,Q1,cash,5.326122e+09,5.326122e+12,THOUSAND,1000.0,1000.0,PRESENTATION_SCALED,ANTM_2021_Q1_FS.xlsx
...,...,...,...,...,...,...,...,...,...,...,...
44412,LAPD,2023,Q4,total_liabilities,1.599281e+11,1.599281e+11,UNIT,1.0,1.0,PRESENTATION_SCALED,LAPD_2023_Q4_FS.xlsx
44418,LAPD,2024,Q1,total_liabilities,1.533519e+11,1.533519e+11,UNIT,1.0,1.0,PRESENTATION_SCALED,LAPD_2024_Q1_FS.xlsx
44424,LAPD,2024,Q2,total_liabilities,1.868606e+11,1.868606e+11,UNIT,1.0,1.0,PRESENTATION_SCALED,LAPD_2024_Q2_FS.xlsx
44430,LAPD,2024,Q4,total_liabilities,1.967051e+11,1.967051e+11,UNIT,1.0,1.0,PRESENTATION_SCALED,LAPD_2024_Q4_FS.xlsx


In [12]:
# CELL 11 - REBUILD RATIOS FOR EXTREME TICKERS ACROSS PERIODS

extreme_ticker_wide_df = (
    neighbor_audit_df
    .pivot_table(
        index=[
            "ticker",
            "year",
            "quarter"
        ],
        columns="metric",
        values="normalized_value",
        aggfunc="first"
    )
    .reset_index()
)

extreme_ticker_wide_df.columns.name = None


extreme_ticker_wide_df[
    "gross_margin"
] = safe_divide(
    extreme_ticker_wide_df.get(
        "gross_profit"
    ),
    extreme_ticker_wide_df.get(
        "revenue"
    )
)


extreme_ticker_wide_df[
    "debt_to_assets"
] = safe_divide(
    extreme_ticker_wide_df.get(
        "total_liabilities"
    ),
    extreme_ticker_wide_df.get(
        "total_assets"
    )
)


extreme_ticker_wide_df[
    "cash_to_assets"
] = safe_divide(
    extreme_ticker_wide_df.get(
        "cash"
    ),
    extreme_ticker_wide_df.get(
        "total_assets"
    )
)


extreme_ticker_wide_df[
    "ocf_to_revenue"
] = safe_divide(
    extreme_ticker_wide_df.get(
        "operating_cash_flow"
    ),
    extreme_ticker_wide_df.get(
        "revenue"
    )
)


display(
    extreme_ticker_wide_df[
        [
            "ticker",
            "year",
            "quarter",
            "gross_margin",
            "debt_to_assets",
            "cash_to_assets",
            "ocf_to_revenue"
        ]
    ]
    .sort_values(
        [
            "ticker",
            "year",
            "quarter"
        ]
    )
)

,ticker,year,quarter,gross_margin,debt_to_assets,cash_to_assets,ocf_to_revenue
0,ANTM,2020,Q1,0.107989,0.414221,0.102715,0.003626
1,ANTM,2020,Q2,0.141885,0.397213,0.100098,0.013624
2,ANTM,2020,Q3,0.161024,0.388870,0.118466,0.061955
3,ANTM,2020,Q4,0.163514,0.399945,0.125574,0.081055
4,ANTM,2021,Q1,0.176486,0.394335,0.162924,0.204138
5,ANTM,2021,Q2,0.183422,0.385716,0.158621,0.139130
6,ANTM,2021,Q3,0.194026,0.389088,0.191220,0.168123
7,ANTM,2021,Q4,0.165404,0.366964,0.154610,0.131164
8,ANTM,2022,Q1,0.251126,0.292824,0.131884,0.049396
9,ANTM,2022,Q2,0.214644,0.334337,0.100249,0.045881


In [13]:
# CELL 12 - INSPECT RAW WORKBOOK CONTEXT FOR SUSPICIOUS RATIO COMPONENTS

from openpyxl import load_workbook


suspicious_cases = [
    {
        "source_file": "ANTM_2023_Q2_FS.xlsx",
        "metrics": [
            "total_assets",
            "total_liabilities",
            "cash"
        ]
    },
    {
        "source_file": "HDTX_2024_Q4_FS.xlsx",
        "metrics": [
            "revenue",
            "gross_profit"
        ]
    },
    {
        "source_file": "LAPD_2022_Q3_FS.xlsx",
        "metrics": [
            "total_assets",
            "total_liabilities"
        ]
    }
]


audit_rows = []


for case in suspicious_cases:

    source_file = case["source_file"]

    case_rows = (
        idx_normalized_df[
            (
                idx_normalized_df["source_file"]
                == source_file
            )
            &
            (
                idx_normalized_df["metric"]
                .isin(case["metrics"])
            )
        ]
        .copy()
    )


    for _, metric_row in case_rows.iterrows():

        wb = None

        try:

            wb = load_workbook(
                _relocated_idx_path(metric_row["source_path"]),
                read_only=True,
                data_only=True
            )

            sheet_name = str(
                metric_row["source_sheet"]
            )

            if sheet_name not in wb.sheetnames:

                audit_rows.append(
                    {
                        "source_file": source_file,
                        "metric": metric_row["metric"],
                        "source_label":
                            metric_row["source_label"],
                        "selected_value":
                            metric_row["selected_value"],
                        "normalized_value":
                            metric_row["normalized_value"],
                        "status": "SHEET_NOT_FOUND",
                        "matches": []
                    }
                )

                continue


            ws = wb[sheet_name]

            target_label = (
                str(metric_row["source_label"])
                .strip()
                .lower()
            )

            matches = []


            for row in ws.iter_rows(
                min_row=1,
                max_row=min(ws.max_row, 300),
                min_col=1,
                max_col=min(ws.max_column, 20),
                values_only=False
            ):

                for cell in row:

                    if cell.value is None:
                        continue

                    cell_text = (
                        str(cell.value)
                        .strip()
                        .lower()
                    )


                    if cell_text == target_label:

                        nearby = []

                        start_col = max(
                            1,
                            cell.column - 1
                        )

                        end_col = min(
                            ws.max_column,
                            cell.column + 6
                        )


                        for col_idx in range(
                            start_col,
                            end_col + 1
                        ):

                            nearby_cell = ws.cell(
                                row=cell.row,
                                column=col_idx
                            )

                            nearby.append(
                                {
                                    "coordinate":
                                        nearby_cell.coordinate,

                                    "value":
                                        nearby_cell.value
                                }
                            )


                        matches.append(
                            {
                                "label_coordinate":
                                    cell.coordinate,

                                "nearby_values":
                                    nearby
                            }
                        )


            audit_rows.append(
                {
                    "source_file":
                        source_file,

                    "ticker":
                        metric_row["ticker"],

                    "year":
                        metric_row["year"],

                    "quarter":
                        metric_row["quarter"],

                    "metric":
                        metric_row["metric"],

                    "source_sheet":
                        sheet_name,

                    "source_label":
                        metric_row["source_label"],

                    "selected_value":
                        metric_row["selected_value"],

                    "normalized_value":
                        metric_row["normalized_value"],

                    "unit_scale":
                        metric_row["unit_scale"],

                    "multiplier":
                        metric_row["multiplier"],

                    "status":
                        (
                            "FOUND_LABEL"
                            if matches
                            else "LABEL_NOT_FOUND"
                        ),

                    "matches":
                        matches
                }
            )


        except Exception as e:

            audit_rows.append(
                {
                    "source_file":
                        source_file,

                    "metric":
                        metric_row["metric"],

                    "source_label":
                        metric_row["source_label"],

                    "selected_value":
                        metric_row["selected_value"],

                    "normalized_value":
                        metric_row["normalized_value"],

                    "status":
                        f"READ_ERROR:{type(e).__name__}",

                    "matches": []
                }
            )


        finally:

            if wb is not None:
                wb.close()


suspicious_workbook_audit_df = pd.DataFrame(
    audit_rows
)


print("AUDIT STATUS")

print(
    suspicious_workbook_audit_df[
        "status"
    ].value_counts(
        dropna=False
    )
)


for _, row in (
    suspicious_workbook_audit_df
    .sort_values(
        [
            "source_file",
            "metric"
        ]
    )
    .iterrows()
):

    print("\n" + "=" * 120)

    print(
        row["source_file"],
        "|",
        row["metric"]
    )

    print(
        "Selected:",
        row["selected_value"],
        "| Normalized:",
        row["normalized_value"]
    )

    print(
        "Label:",
        row["source_label"]
    )

    print(
        "Unit:",
        row.get("unit_scale"),
        "| Multiplier:",
        row.get("multiplier")
    )

    print(
        "Workbook matches:"
    )

    print(
        row["matches"]
    )

AUDIT STATUS
status
FOUND_LABEL    7
Name: count, dtype: int64

ANTM_2023_Q2_FS.xlsx | cash
Selected: 6581605.0 | Normalized: 6581605000000.0
Label: Kas dan setara kas
Unit: MILLION | Multiplier: 1000000.0
Workbook matches:
[{'label_coordinate': 'A7', 'nearby_values': [{'coordinate': 'A7', 'value': 'Kas dan setara kas'}, {'coordinate': 'B7', 'value': 6581605.0}, {'coordinate': 'C7', 'value': 4476491.0}, {'coordinate': 'D7', 'value': 'Cash and cash equivalents'}]}]

ANTM_2023_Q2_FS.xlsx | total_assets
Selected: 36.368666 | Normalized: 36368666.0
Label: Jumlah aset
Unit: MILLION | Multiplier: 1000000.0
Workbook matches:
[{'label_coordinate': 'A127', 'nearby_values': [{'coordinate': 'A127', 'value': 'Jumlah aset'}, {'coordinate': 'B127', 'value': 36.368666}, {'coordinate': 'C127', 'value': 33637271.0}, {'coordinate': 'D127', 'value': 'Total assets'}]}]

ANTM_2023_Q2_FS.xlsx | total_liabilities
Selected: 12692580.0 | Normalized: 12692580000000.0
Label: Jumlah liabilitas
Unit: MILLION | Mul

In [14]:
# CELL 13 - PRECISE CELL AUDIT FOR SUSPICIOUS BENCHMARK VALUES

from openpyxl import load_workbook


precise_checks = [
    {
        "source_file": "ANTM_2023_Q2_FS.xlsx",
        "sheet": "1210000",
        "cells": ["B127", "C127"],
        "description": "ANTM total assets"
    },
    {
        "source_file": "LAPD_2022_Q3_FS.xlsx",
        "sheet": "3210000",
        "cells": ["B70", "C70", "B152", "C152"],
        "description": "LAPD assets and liabilities"
    },
    {
        "source_file": "HDTX_2024_Q4_FS.xlsx",
        "sheet": "1311000",
        "cells": ["A6", "B6", "C6", "D6", "A8", "B8", "C8", "D8"],
        "description": "HDTX revenue and gross profit"
    }
]


for check in precise_checks:

    source_file = check["source_file"]

    source_path = (
        idx_normalized_df.loc[
            idx_normalized_df["source_file"] == source_file,
            "source_path"
        ]
        .iloc[0]
    )

    print("\n" + "=" * 120)
    print(source_file)
    print(check["description"])

    # data_only=True -> value displayed/calculated by workbook
    wb_values = load_workbook(
        _relocated_idx_path(source_path),
        read_only=False,
        data_only=True
    )

    # data_only=False -> formula/raw workbook content
    wb_raw = load_workbook(
        _relocated_idx_path(source_path),
        read_only=False,
        data_only=False
    )

    try:

        ws_values = wb_values[check["sheet"]]
        ws_raw = wb_raw[check["sheet"]]

        for coordinate in check["cells"]:

            value_cell = ws_values[coordinate]
            raw_cell = ws_raw[coordinate]

            print(
                coordinate,
                "| value:",
                repr(value_cell.value),
                "| raw/formula:",
                repr(raw_cell.value),
                "| number_format:",
                repr(value_cell.number_format)
            )

    finally:

        wb_values.close()
        wb_raw.close()


ANTM_2023_Q2_FS.xlsx
ANTM total assets
B127 | value: 36.368666 | raw/formula: 36.368666 | number_format: '#,##0;(#,##0)'
C127 | value: 33637271.0 | raw/formula: 33637271.0 | number_format: '#,##0;(#,##0)'

LAPD_2022_Q3_FS.xlsx
LAPD assets and liabilities
B70 | value: 69169231.0 | raw/formula: 69169231.0 | number_format: '#,##0;\\(#,##0\\)'
C70 | value: 77939737.0 | raw/formula: 77939737.0 | number_format: '#,##0;\\(#,##0\\)'
B152 | value: 262341998765.0 | raw/formula: 262341998765.0 | number_format: '#,##0;\\(#,##0\\)'
C152 | value: 248715057382.0 | raw/formula: 248715057382.0 | number_format: '#,##0;\\(#,##0\\)'

HDTX_2024_Q4_FS.xlsx
HDTX revenue and gross profit


e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


A6 | value: 'Penjualan dan pendapatan usaha' | raw/formula: 'Penjualan dan pendapatan usaha' | number_format: 'General'
B6 | value: 538 | raw/formula: 538 | number_format: '#,##0;\\(#,##0\\)'
C6 | value: 27908 | raw/formula: 27908 | number_format: '#,##0;\\(#,##0\\)'
D6 | value: 'Sales and revenue' | raw/formula: 'Sales and revenue' | number_format: 'General'
A8 | value: 'Jumlah laba bruto' | raw/formula: 'Jumlah laba bruto' | number_format: 'General'
B8 | value: -30710637 | raw/formula: -30710637 | number_format: '#,##0;\\(#,##0\\)'
C8 | value: -29480676 | raw/formula: -29480676 | number_format: '#,##0;\\(#,##0\\)'
D8 | value: 'Total gross profit' | raw/formula: 'Total gross profit' | number_format: 'General'


In [15]:
# CELL 14 - COMPARE SUSPICIOUS COMPONENTS ACROSS NEIGHBOR PERIODS

component_checks = {
    "ANTM": [
        "total_assets",
        "total_liabilities",
        "cash"
    ],

    "LAPD": [
        "total_assets",
        "total_liabilities",
        "cash"
    ],

    "HDTX": [
        "revenue",
        "gross_profit"
    ]
}


component_audit_rows = []


for ticker, metrics in component_checks.items():

    subset = (
        idx_normalized_df[
            (idx_normalized_df["ticker"] == ticker)
            &
            (idx_normalized_df["metric"].isin(metrics))
        ]
        .copy()
    )

    subset["quarter_order"] = subset["quarter"].map(
        {
            "Q1": 1,
            "Q2": 2,
            "Q3": 3,
            "Q4": 4
        }
    )

    subset = subset.sort_values(
        [
            "metric",
            "year",
            "quarter_order"
        ]
    )

    component_audit_rows.append(subset)


suspicious_component_history_df = pd.concat(
    component_audit_rows,
    ignore_index=True
)


display(
    suspicious_component_history_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "normalized_value",
            "unit_scale",
            "effective_multiplier",
            "source_file"
        ]
    ]
)

,ticker,year,quarter,metric,selected_value,normalized_value,unit_scale,effective_multiplier,source_file
0,ANTM,2020,Q1,cash,3.160647e+09,3.160647e+12,THOUSAND,1000.0,ANTM_2020_Q1_FS.xlsx
1,ANTM,2020,Q2,cash,3.006270e+09,3.006270e+12,THOUSAND,1000.0,ANTM_2020_Q2_FS.xlsx
2,ANTM,2020,Q3,cash,3.669383e+09,3.669383e+12,THOUSAND,1000.0,ANTM_2020_Q3_FS.xlsx
3,ANTM,2020,Q4,cash,3.984388e+09,3.984388e+12,THOUSAND,1000.0,ANTM_2020_Q4_FS.xlsx
4,ANTM,2021,Q1,cash,5.326122e+09,5.326122e+12,THOUSAND,1000.0,ANTM_2021_Q1_FS.xlsx
...,...,...,...,...,...,...,...,...,...
155,HDTX,2023,Q4,revenue,2.790800e+04,2.790800e+07,THOUSAND,1000.0,HDTX_2023_Q4_FS.xlsx
156,HDTX,2024,Q1,revenue,5.380000e+02,5.380000e+05,THOUSAND,1000.0,HDTX_2024_Q1_FS.xlsx
157,HDTX,2024,Q3,revenue,5.380000e+02,5.380000e+05,THOUSAND,1000.0,HDTX_2024_Q3_FS.xlsx
158,HDTX,2024,Q4,revenue,5.380000e+02,5.380000e+05,THOUSAND,1000.0,HDTX_2024_Q4_FS.xlsx


In [16]:
# CELL 15 - GLOBAL TEMPORAL ANOMALY SCAN FOR NORMALIZED METRICS

global_metric_audit_df = (
    idx_normalized_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "normalized_value",
            "source_file"
        ]
    ]
    .copy()
)


quarter_order = {
    "Q1": 1,
    "Q2": 2,
    "Q3": 3,
    "Q4": 4
}


global_metric_audit_df[
    "quarter_order"
] = (
    global_metric_audit_df[
        "quarter"
    ].map(quarter_order)
)


global_metric_audit_df = (
    global_metric_audit_df
    .sort_values(
        [
            "ticker",
            "metric",
            "year",
            "quarter_order"
        ]
    )
    .reset_index(drop=True)
)


# Absolute magnitude in log space.
# +1 avoids problems when value = 0.
global_metric_audit_df[
    "log_abs_value"
] = np.log10(
    global_metric_audit_df[
        "normalized_value"
    ].abs() + 1
)


global_metric_audit_df[
    "previous_log_abs_value"
] = (
    global_metric_audit_df
    .groupby(
        [
            "ticker",
            "metric"
        ]
    )[
        "log_abs_value"
    ]
    .shift(1)
)


global_metric_audit_df[
    "log_change_from_previous"
] = (
    global_metric_audit_df[
        "log_abs_value"
    ]
    -
    global_metric_audit_df[
        "previous_log_abs_value"
    ]
)


global_metric_audit_df[
    "abs_log_change"
] = (
    global_metric_audit_df[
        "log_change_from_previous"
    ].abs()
)


print(
    "Metric rows:",
    len(global_metric_audit_df)
)

print(
    "Rows with temporal comparison:",
    global_metric_audit_df[
        "abs_log_change"
    ].notna().sum()
)


print("\nTEMPORAL CHANGE SUMMARY")

print(
    global_metric_audit_df[
        "abs_log_change"
    ].describe(
        percentiles=[
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

Metric rows: 83442
Rows with temporal comparison: 78139

TEMPORAL CHANGE SUMMARY
count    78139.000000
mean         0.233615
std          0.382422
min          0.000000
90%          0.600754
95%          0.801132
99%          1.530549
99.5%        1.950786
99.9%        4.155754
max         11.328295
Name: abs_log_change, dtype: float64


In [17]:
# CELL 16 - FLAG GLOBAL EXTREME TEMPORAL CHANGES

temporal_threshold = (
    global_metric_audit_df[
        "abs_log_change"
    ]
    .quantile(0.995)
)


global_metric_audit_df[
    "temporal_anomaly"
] = (
    global_metric_audit_df[
        "abs_log_change"
    ]
    >= temporal_threshold
)


print(
    "99.5% temporal anomaly threshold:",
    temporal_threshold
)

print(
    "Flagged metric rows:",
    global_metric_audit_df[
        "temporal_anomaly"
    ].sum()
)

print(
    "Affected source files:",
    global_metric_audit_df.loc[
        global_metric_audit_df[
            "temporal_anomaly"
        ],
        "source_file"
    ].nunique()
)

print(
    "Affected tickers:",
    global_metric_audit_df.loc[
        global_metric_audit_df[
            "temporal_anomaly"
        ],
        "ticker"
    ].nunique()
)


display(
    global_metric_audit_df[
        global_metric_audit_df[
            "temporal_anomaly"
        ]
    ][
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "normalized_value",
            "log_change_from_previous",
            "abs_log_change",
            "source_file"
        ]
    ]
    .sort_values(
        "abs_log_change",
        ascending=False
    )
    .head(100)
)

99.5% temporal anomaly threshold: 1.950785662694603
Flagged metric rows: 391
Affected source files: 296
Affected tickers: 193


,ticker,year,quarter,metric,normalized_value,log_change_from_previous,abs_log_change,source_file
53341,MTPS,2022,Q1,gross_profit,0.000000e+00,-11.328295,11.328295,MTPS_2022_Q1_FS.xlsx
53378,MTPS,2022,Q1,revenue,0.000000e+00,-10.739593,10.739593,MTPS_2022_Q1_FS.xlsx
26458,ETWA,2023,Q1,revenue,0.000000e+00,-10.684009,10.684009,ETWA_2023_Q1_FS.xlsx
44398,LAPD,2023,Q3,revenue,4.813373e+10,10.682449,10.682449,LAPD_2023_Q3_FS.xlsx
75319,TGRA,2022,Q1,revenue,0.000000e+00,-10.587601,10.587601,TGRA_2022_Q1_FS.xlsx
...,...,...,...,...,...,...,...,...
21206,DEWA,2023,Q1,revenue,1.761394e+12,3.636449,3.636449,DEWA_2023_Q1_FS.xlsx
7214,ATLA,2024,Q4,operating_cash_flow,-5.452371e+10,3.602214,3.602214,ATLA_2024_Q4_FS.xlsx
48137,MBSS,2024,Q1,operating_cash_flow,6.558728e+10,3.596629,3.596629,MBSS_2024_Q1_FS.xlsx
53726,MYOH,2024,Q3,operating_cash_flow,3.043933e+07,-3.584068,3.584068,MYOH_2024_Q3_FS.xlsx


In [18]:
# CELL 17 - CHECK KNOWN CASES AND GLOBAL ANOMALY DISTRIBUTION

known_suspicious_tickers = [
    "ANTM",
    "LAPD",
    "HDTX"
]


print("KNOWN CASES")

display(
    global_metric_audit_df[
        (
            global_metric_audit_df[
                "ticker"
            ].isin(
                known_suspicious_tickers
            )
        )
        &
        (
            global_metric_audit_df[
                "temporal_anomaly"
            ]
        )
    ][
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "normalized_value",
            "abs_log_change",
            "temporal_anomaly",
            "source_file"
        ]
    ]
    .sort_values(
        [
            "ticker",
            "year",
            "quarter",
            "metric"
        ]
    )
)


print("\nMOST AFFECTED TICKERS")

print(
    global_metric_audit_df.loc[
        global_metric_audit_df[
            "temporal_anomaly"
        ],
        "ticker"
    ]
    .value_counts()
    .head(50)
)


print("\nANOMALIES BY METRIC")

print(
    global_metric_audit_df.loc[
        global_metric_audit_df[
            "temporal_anomaly"
        ],
        "metric"
    ]
    .value_counts()
)

KNOWN CASES


,ticker,year,quarter,metric,normalized_value,abs_log_change,temporal_anomaly,source_file
4190,ANTM,2023,Q2,total_assets,3.636867e+07,5.982517,True,ANTM_2023_Q2_FS.xlsx
4191,ANTM,2023,Q3,total_assets,3.550017e+13,5.989503,True,ANTM_2023_Q3_FS.xlsx
31654,HDTX,2023,Q1,revenue,5.402000e+06,3.046012,True,HDTX_2023_Q1_FS.xlsx
31641,HDTX,2025,Q1,operating_cash_flow,2.321100e+07,2.186546,True,HDTX_2025_Q1_FS.xlsx
31661,HDTX,2025,Q1,revenue,0.000000e+00,5.730783,True,HDTX_2025_Q1_FS.xlsx
44388,LAPD,2020,Q2,revenue,2.873165e+09,9.458361,True,LAPD_2020_Q2_FS.xlsx
44391,LAPD,2021,Q1,revenue,8.858115e+06,3.204115,True,LAPD_2021_Q1_FS.xlsx
44393,LAPD,2021,Q3,revenue,0.000000e+00,6.947341,True,LAPD_2021_Q3_FS.xlsx
44411,LAPD,2021,Q4,total_assets,7.793900e+07,3.092273,True,LAPD_2021_Q4_FS.xlsx
44357,LAPD,2022,Q1,gross_profit,0.000000e+00,10.425343,True,LAPD_2022_Q1_FS.xlsx



MOST AFFECTED TICKERS
ticker
SGER    12
LAPD     9
ALMI     8
ARGO     8
MYOH     8
MTPS     7
AIMS     6
CUAN     6
DEWA     6
IMJS     6
KOBX     6
MBSS     6
RONY     6
SMCB     6
SMMT     6
ZBRA     6
COCO     5
HDIT     5
BTON     4
FIMP     4
GTBO     4
MAPA     4
MGNA     4
PSDN     4
TRGU     4
AKKU     3
BACA     3
BAPI     3
BIKE     3
CSMI     3
HDTX     3
KOKA     3
LABA     3
NASI     3
NPGF     3
TIRT     3
AGRS     2
AISA     2
AMAG     2
AMAR     2
ANTM     2
APLN     2
ATAP     2
BBSS     2
BDMN     2
BEEF     2
BESS     2
BMAS     2
BOLT     2
DMND     2
Name: count, dtype: int64

ANOMALIES BY METRIC
metric
operating_cash_flow    193
gross_profit            64
revenue                 48
cash                    47
total_assets            20
total_liabilities       19
Name: count, dtype: int64


In [19]:
# CELL 15 - BUILD GLOBAL METRIC QUALITY FLAGS
# We do NOT alter source values.
# We only flag suspicious observations for benchmark usage.

benchmark_quality_df = (
    idx_normalized_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "normalized_value",
            "source_file"
        ]
    ]
    .copy()
)


quarter_order = {
    "Q1": 1,
    "Q2": 2,
    "Q3": 3,
    "Q4": 4
}


benchmark_quality_df[
    "quarter_order"
] = (
    benchmark_quality_df[
        "quarter"
    ].map(
        quarter_order
    )
)


benchmark_quality_df = (
    benchmark_quality_df
    .sort_values(
        [
            "ticker",
            "metric",
            "year",
            "quarter_order"
        ]
    )
    .reset_index(drop=True)
)


# Previous and next normalized values
grouped = (
    benchmark_quality_df
    .groupby(
        [
            "ticker",
            "metric"
        ]
    )
)


benchmark_quality_df[
    "prev_value"
] = (
    grouped[
        "normalized_value"
    ].shift(1)
)


benchmark_quality_df[
    "next_value"
] = (
    grouped[
        "normalized_value"
    ].shift(-1)
)


def log_distance(a, b):

    if (
        pd.isna(a)
        or pd.isna(b)
    ):
        return np.nan

    a = abs(float(a))
    b = abs(float(b))

    if a == 0 or b == 0:
        return np.nan

    return abs(
        np.log10(
            a / b
        )
    )


def detect_temporal_anomaly(row):

    current = row[
        "normalized_value"
    ]

    if pd.isna(current):
        return False


    distances = []

    if pd.notna(row["prev_value"]):

        d = log_distance(
            current,
            row["prev_value"]
        )

        if pd.notna(d):
            distances.append(d)


    if pd.notna(row["next_value"]):

        d = log_distance(
            current,
            row["next_value"]
        )

        if pd.notna(d):
            distances.append(d)


    # No temporal evidence
    if not distances:
        return False


    # 2 log10 = factor 100 difference.
    # We only FLAG it, not correct it.
    return min(distances) >= 2


benchmark_quality_df[
    "temporal_anomaly"
] = (
    benchmark_quality_df.apply(
        detect_temporal_anomaly,
        axis=1
    )
)


print(
    "Total metric rows:",
    len(benchmark_quality_df)
)

print(
    "Temporal anomaly rows:",
    benchmark_quality_df[
        "temporal_anomaly"
    ].sum()
)

print(
    "Affected tickers:",
    benchmark_quality_df.loc[
        benchmark_quality_df[
            "temporal_anomaly"
        ],
        "ticker"
    ].nunique()
)


print("\nANOMALIES BY METRIC")

print(
    benchmark_quality_df.loc[
        benchmark_quality_df[
            "temporal_anomaly"
        ],
        "metric"
    ].value_counts()
)

Total metric rows: 83442
Temporal anomaly rows: 115
Affected tickers: 80

ANOMALIES BY METRIC
metric
operating_cash_flow    65
cash                   19
gross_profit           13
revenue                 6
total_assets            6
total_liabilities       6
Name: count, dtype: int64


In [22]:
# CELL 16 - APPLY QUALITY FLAGS TO FINANCIAL RATIOS
# FIXED VERSION:
# A ratio is valid only if:
# 1. the ratio value exists, and
# 2. none of its required metrics are flagged as temporal anomalies.

metric_quality_wide_df = (
    benchmark_quality_df
    .pivot_table(
        index=[
            "ticker",
            "year",
            "quarter"
        ],
        columns="metric",
        values="temporal_anomaly",
        aggfunc="max"
    )
    .reset_index()
)

metric_quality_wide_df.columns.name = None


metric_quality_wide_df = (
    metric_quality_wide_df
    .rename(
        columns={
            "gross_profit":
                "gross_profit_anomaly",

            "revenue":
                "revenue_anomaly",

            "total_liabilities":
                "total_liabilities_anomaly",

            "total_assets":
                "total_assets_anomaly",

            "cash":
                "cash_anomaly",

            "operating_cash_flow":
                "operating_cash_flow_anomaly"
        }
    )
)


benchmark_final_df = (
    benchmark_ratio_df
    .merge(
        metric_quality_wide_df,
        on=[
            "ticker",
            "year",
            "quarter"
        ],
        how="left",
        validate="one_to_one"
    )
)


anomaly_columns = [
    "gross_profit_anomaly",
    "revenue_anomaly",
    "total_liabilities_anomaly",
    "total_assets_anomaly",
    "cash_anomaly",
    "operating_cash_flow_anomaly"
]


for column in anomaly_columns:

    if column not in benchmark_final_df.columns:
        benchmark_final_df[column] = False

    benchmark_final_df[column] = (
        benchmark_final_df[
            column
        ]
        .fillna(False)
        .astype(bool)
    )


# --------------------------------------------------
# RATIO-SPECIFIC QUALITY FLAGS
# --------------------------------------------------

benchmark_final_df[
    "gross_margin_valid"
] = (
    benchmark_final_df[
        "gross_margin"
    ].notna()
    &
    ~(
        benchmark_final_df[
            "gross_profit_anomaly"
        ]
        |
        benchmark_final_df[
            "revenue_anomaly"
        ]
    )
)


benchmark_final_df[
    "debt_to_assets_valid"
] = (
    benchmark_final_df[
        "debt_to_assets"
    ].notna()
    &
    ~(
        benchmark_final_df[
            "total_liabilities_anomaly"
        ]
        |
        benchmark_final_df[
            "total_assets_anomaly"
        ]
    )
)


benchmark_final_df[
    "cash_to_assets_valid"
] = (
    benchmark_final_df[
        "cash_to_assets"
    ].notna()
    &
    ~(
        benchmark_final_df[
            "cash_anomaly"
        ]
        |
        benchmark_final_df[
            "total_assets_anomaly"
        ]
    )
)


benchmark_final_df[
    "ocf_to_revenue_valid"
] = (
    benchmark_final_df[
        "ocf_to_revenue"
    ].notna()
    &
    ~(
        benchmark_final_df[
            "operating_cash_flow_anomaly"
        ]
        |
        benchmark_final_df[
            "revenue_anomaly"
        ]
    )
)


print("VALID RATIO COUNTS")

for ratio in [
    "gross_margin",
    "debt_to_assets",
    "cash_to_assets",
    "ocf_to_revenue"
]:

    valid_col = f"{ratio}_valid"

    available_count = (
        benchmark_final_df[
            ratio
        ]
        .notna()
        .sum()
    )

    valid_count = (
        benchmark_final_df[
            valid_col
        ]
        .sum()
    )

    excluded_count = (
        available_count
        - valid_count
    )

    print(
        f"{ratio}: "
        f"{valid_count} valid / "
        f"{available_count} available / "
        f"{excluded_count} excluded"
    )

VALID RATIO COUNTS
gross_margin: 12332 valid / 12349 available / 17 excluded
debt_to_assets: 14746 valid / 14753 available / 7 excluded
cash_to_assets: 13758 valid / 13778 available / 20 excluded
ocf_to_revenue: 12910 valid / 12963 available / 53 excluded


In [23]:
# CELL 17 - BUILD QUALITY-CONTROLLED RATIO DATASET

quality_ratio_df = (
    benchmark_final_df[
        [
            "ticker",
            "year",
            "quarter",
            "gross_margin",
            "gross_margin_valid",
            "debt_to_assets",
            "debt_to_assets_valid",
            "cash_to_assets",
            "cash_to_assets_valid",
            "ocf_to_revenue",
            "ocf_to_revenue_valid"
        ]
    ]
    .copy()
)


# Create benchmark-safe versions.
# Invalid observations remain in the dataframe for provenance,
# but are excluded from benchmark statistics by becoming NaN.

for ratio in [
    "gross_margin",
    "debt_to_assets",
    "cash_to_assets",
    "ocf_to_revenue"
]:

    valid_column = f"{ratio}_valid"
    benchmark_column = f"{ratio}_benchmark"

    quality_ratio_df[
        benchmark_column
    ] = np.where(
        quality_ratio_df[
            valid_column
        ],
        quality_ratio_df[
            ratio
        ],
        np.nan
    )


benchmark_ratio_columns = [
    "gross_margin_benchmark",
    "debt_to_assets_benchmark",
    "cash_to_assets_benchmark",
    "ocf_to_revenue_benchmark"
]


print("BENCHMARK-SAFE RATIO AVAILABILITY")

for column in benchmark_ratio_columns:

    print(
        column,
        ":",
        quality_ratio_df[
            column
        ].notna().sum()
    )


print("\nBENCHMARK-SAFE DISTRIBUTION")

for column in benchmark_ratio_columns:

    print("\n", column)

    print(
        quality_ratio_df[
            column
        ].describe(
            percentiles=[
                0.05,
                0.25,
                0.50,
                0.75,
                0.95
            ]
        )
    )

BENCHMARK-SAFE RATIO AVAILABILITY
gross_margin_benchmark : 12332
debt_to_assets_benchmark : 14746
cash_to_assets_benchmark : 13758
ocf_to_revenue_benchmark : 12910

BENCHMARK-SAFE DISTRIBUTION

 gross_margin_benchmark
count    12332.000000
mean        -8.940889
std        608.058487
min     -57082.968401
5%          -0.008175
25%          0.125768
50%          0.251172
75%          0.437408
95%          0.701815
max          1.142575
Name: gross_margin_benchmark, dtype: float64

 debt_to_assets_benchmark
count    14746.000000
mean         2.690056
std         72.399953
min          0.000008
5%           0.069907
25%          0.255347
50%          0.450176
75%          0.660683
95%          1.019874
max       3792.755752
Name: debt_to_assets_benchmark, dtype: float64

 cash_to_assets_benchmark
count    1.375800e+04
mean     9.706857e-02
std      1.229513e-01
min      1.151412e-07
5%       1.781631e-03
25%      1.509420e-02
50%      5.236307e-02
75%      1.336019e-01
95%      3.270472e-0

In [24]:
# CELL 18 - CALCULATE OVERALL IDX BENCHMARK STATISTICS

overall_benchmark_rows = []


ratio_mapping = {
    "gross_margin":
        "gross_margin_benchmark",

    "debt_to_assets":
        "debt_to_assets_benchmark",

    "cash_to_assets":
        "cash_to_assets_benchmark",

    "ocf_to_revenue":
        "ocf_to_revenue_benchmark",
}


for ratio_name, column_name in ratio_mapping.items():

    values = (
        quality_ratio_df[
            column_name
        ]
        .dropna()
    )


    overall_benchmark_rows.append(
        {
            "benchmark_scope":
                "IDX_ALL",

            "ratio":
                ratio_name,

            "count":
                len(values),

            "p05":
                values.quantile(0.05),

            "p25":
                values.quantile(0.25),

            "median":
                values.quantile(0.50),

            "p75":
                values.quantile(0.75),

            "p95":
                values.quantile(0.95),

            "mean":
                values.mean(),

            "std":
                values.std(),
        }
    )


overall_benchmark_df = pd.DataFrame(
    overall_benchmark_rows
)


display(
    overall_benchmark_df
)

,benchmark_scope,ratio,count,p05,p25,median,p75,p95,mean,std
0,IDX_ALL,gross_margin,12332,-0.008175,0.125768,0.251172,0.437408,0.701815,-8.940889,608.058487
1,IDX_ALL,debt_to_assets,14746,0.069907,0.255347,0.450176,0.660683,1.019874,2.690056,72.399953
2,IDX_ALL,cash_to_assets,13758,0.001782,0.015094,0.052363,0.133602,0.327047,0.097069,0.122951
3,IDX_ALL,ocf_to_revenue,12910,-0.891228,-0.044744,0.061471,0.199658,0.558679,-1.500199,91.152830


In [25]:
# CELL 19 - CALCULATE IDX BENCHMARK BY YEAR AND QUARTER

period_benchmark_rows = []


for (
    year,
    quarter
), group in quality_ratio_df.groupby(
    [
        "year",
        "quarter"
    ]
):

    for ratio_name, column_name in ratio_mapping.items():

        values = (
            group[
                column_name
            ]
            .dropna()
        )


        if len(values) == 0:
            continue


        period_benchmark_rows.append(
            {
                "benchmark_scope":
                    "IDX_PERIOD",

                "year":
                    year,

                "quarter":
                    quarter,

                "ratio":
                    ratio_name,

                "count":
                    len(values),

                "p05":
                    values.quantile(0.05),

                "p25":
                    values.quantile(0.25),

                "median":
                    values.quantile(0.50),

                "p75":
                    values.quantile(0.75),

                "p95":
                    values.quantile(0.95),

                "mean":
                    values.mean(),

                "std":
                    values.std(),
            }
        )


period_benchmark_df = pd.DataFrame(
    period_benchmark_rows
)


print(
    "Period benchmark rows:",
    len(period_benchmark_df)
)


display(
    period_benchmark_df
    .sort_values(
        [
            "year",
            "quarter",
            "ratio"
        ]
    )
    .head(50)
)

Period benchmark rows: 84


,benchmark_scope,year,quarter,ratio,count,p05,p25,median,p75,p95,mean,std
2,IDX_PERIOD,2020,Q1,cash_to_assets,590,0.001767,0.014940,0.049326,0.116306,0.292905,0.089435,0.118615
1,IDX_PERIOD,2020,Q1,debt_to_assets,633,0.075756,0.288474,0.493859,0.697972,0.986329,2.185317,38.429911
0,IDX_PERIOD,2020,Q1,gross_margin,521,-0.088255,0.117811,0.231797,0.429462,0.720901,0.224286,0.740803
3,IDX_PERIOD,2020,Q1,ocf_to_revenue,546,-1.069715,-0.066307,0.065453,0.232602,0.718511,-0.101741,4.112210
6,IDX_PERIOD,2020,Q2,cash_to_assets,603,0.001535,0.012750,0.043359,0.114903,0.297498,0.087169,0.119216
5,IDX_PERIOD,2020,Q2,debt_to_assets,648,0.079406,0.271885,0.484249,0.678927,0.973803,2.147498,37.284593
4,IDX_PERIOD,2020,Q2,gross_margin,535,-0.139149,0.110616,0.220658,0.401127,0.705571,0.180686,1.232400
7,IDX_PERIOD,2020,Q2,ocf_to_revenue,564,-1.005428,-0.070397,0.059895,0.205744,0.599636,-0.331865,3.965024
10,IDX_PERIOD,2020,Q3,cash_to_assets,609,0.001643,0.012859,0.044257,0.113060,0.293245,0.086861,0.119986
9,IDX_PERIOD,2020,Q3,debt_to_assets,654,0.080322,0.280957,0.486065,0.687861,0.988160,3.963350,83.264821


In [26]:
# CELL 20 - BUILD FINAL IDX BENCHMARK TABLES

final_overall_benchmark_df = (
    overall_benchmark_df[
        [
            "benchmark_scope",
            "ratio",
            "count",
            "p05",
            "p25",
            "median",
            "p75",
            "p95"
        ]
    ]
    .copy()
)


final_period_benchmark_df = (
    period_benchmark_df[
        [
            "benchmark_scope",
            "year",
            "quarter",
            "ratio",
            "count",
            "p05",
            "p25",
            "median",
            "p75",
            "p95"
        ]
    ]
    .copy()
)


print("FINAL OVERALL BENCHMARK")

display(
    final_overall_benchmark_df
)


print("\nFINAL PERIOD BENCHMARK ROWS")

print(
    len(final_period_benchmark_df)
)


display(
    final_period_benchmark_df
    .sort_values(
        [
            "year",
            "quarter",
            "ratio"
        ]
    )
    .head(40)
)

FINAL OVERALL BENCHMARK


,benchmark_scope,ratio,count,p05,p25,median,p75,p95
0,IDX_ALL,gross_margin,12332,-0.008175,0.125768,0.251172,0.437408,0.701815
1,IDX_ALL,debt_to_assets,14746,0.069907,0.255347,0.450176,0.660683,1.019874
2,IDX_ALL,cash_to_assets,13758,0.001782,0.015094,0.052363,0.133602,0.327047
3,IDX_ALL,ocf_to_revenue,12910,-0.891228,-0.044744,0.061471,0.199658,0.558679



FINAL PERIOD BENCHMARK ROWS
84


,benchmark_scope,year,quarter,ratio,count,p05,p25,median,p75,p95
2,IDX_PERIOD,2020,Q1,cash_to_assets,590,0.001767,0.014940,0.049326,0.116306,0.292905
1,IDX_PERIOD,2020,Q1,debt_to_assets,633,0.075756,0.288474,0.493859,0.697972,0.986329
0,IDX_PERIOD,2020,Q1,gross_margin,521,-0.088255,0.117811,0.231797,0.429462,0.720901
3,IDX_PERIOD,2020,Q1,ocf_to_revenue,546,-1.069715,-0.066307,0.065453,0.232602,0.718511
6,IDX_PERIOD,2020,Q2,cash_to_assets,603,0.001535,0.012750,0.043359,0.114903,0.297498
5,IDX_PERIOD,2020,Q2,debt_to_assets,648,0.079406,0.271885,0.484249,0.678927,0.973803
4,IDX_PERIOD,2020,Q2,gross_margin,535,-0.139149,0.110616,0.220658,0.401127,0.705571
7,IDX_PERIOD,2020,Q2,ocf_to_revenue,564,-1.005428,-0.070397,0.059895,0.205744,0.599636
10,IDX_PERIOD,2020,Q3,cash_to_assets,609,0.001643,0.012859,0.044257,0.113060,0.293245
9,IDX_PERIOD,2020,Q3,debt_to_assets,654,0.080322,0.280957,0.486065,0.687861,0.988160


In [27]:
# CELL 21 - SAVE FINAL IDX BENCHMARK DATASETS

FINAL_RATIO_FILE = Path(
    "data/idx_financial/benchmark/idx_financial_ratio_benchmark_ready.csv"
)

FINAL_OVERALL_BENCHMARK_FILE = Path(
    "data/idx_financial/benchmark/idx_financial_benchmark_overall.csv"
)

FINAL_PERIOD_BENCHMARK_FILE = Path(
    "data/idx_financial/benchmark/idx_financial_benchmark_by_period.csv"
)


quality_ratio_df.to_csv(
    FINAL_RATIO_FILE,
    index=False
)


final_overall_benchmark_df.to_csv(
    FINAL_OVERALL_BENCHMARK_FILE,
    index=False
)


final_period_benchmark_df.to_csv(
    FINAL_PERIOD_BENCHMARK_FILE,
    index=False
)


print(
    "Saved ratio-level benchmark-ready data:",
    FINAL_RATIO_FILE
)

print(
    "Saved overall benchmark:",
    FINAL_OVERALL_BENCHMARK_FILE
)

print(
    "Saved period benchmark:",
    FINAL_PERIOD_BENCHMARK_FILE
)


print("\nROWS")

print(
    "Ratio-level rows:",
    len(quality_ratio_df)
)

print(
    "Overall benchmark rows:",
    len(final_overall_benchmark_df)
)

print(
    "Period benchmark rows:",
    len(final_period_benchmark_df)
)

Saved ratio-level benchmark-ready data: data\idx_financial_ratio_benchmark_ready.csv
Saved overall benchmark: data\idx_financial_benchmark_overall.csv
Saved period benchmark: data\idx_financial_benchmark_by_period.csv

ROWS
Ratio-level rows: 14753
Overall benchmark rows: 4
Period benchmark rows: 84
